## Document Loading

In [2]:
from langchain_core.documents import Document

In [ ]:
# Creating a documnet object
doc = Document(
                page_content="Hello everyone, welcome the RAG from Scratch exercise",
                metadata={
                    "source": "rag.txt",
                    "page": 1,
                    "author": "Serge Lawson",
                    "date_created": "13 March 2026"
                }
            )

In [4]:
doc

Document(metadata={'source': 'rag.txt', 'page': 1, 'author': 'Serge Lawson', 'date_created': '13 March 2026'}, page_content='Hello everyone, welcome the RAG from Scratch exercise')

In [ ]:
from langchain_community.document_loaders import TextLoader

# Loading text from txt file
loader = TextLoader("../data/compiler.txt")
document = loader.load()
document


[Document(metadata={'source': '../data/compiler.txt'}, page_content='# Compilers Overview\n\nA compiler is a special program that translates source code written in a high-level programming language into machine code, bytecode, or another programming language.\n\n## Key Stages of Compilation:\n\n1. **Lexical Analysis** - Breaking source code into tokens\n2. **Syntax Analysis** - Parsing tokens into an abstract syntax tree (AST)\n3. **Semantic Analysis** - Checking for semantic errors and type consistency\n4. **Intermediate Code Generation** - Creating an intermediate representation\n5. **Optimization** - Improving code efficiency\n6. **Code Generation** - Producing target machine code\n\n## Types of Compilers:\n\n- **Single-pass compilers** - Process source code in one pass\n- **Multi-pass compilers** - Process source code multiple times\n- **Just-in-Time (JIT) compilers** - Compile during program execution\n- **Cross-compilers** - Generate code for a different platform\n\n## Popular Co

In [10]:
from langchain_community.document_loaders import DirectoryLoader
# Loading all text files from directory

dir_loader = DirectoryLoader(
    "../data/", # Path to directory
    glob="**/*.txt", # Pattern matcher
    loader_cls=TextLoader, # Loader Class
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    )

documents = dir_loader.load()
documents

100%|██████████| 2/2 [00:00<00:00, 625.41it/s]


[Document(metadata={'source': '../data/compiler.txt'}, page_content='# Compilers Overview\n\nA compiler is a special program that translates source code written in a high-level programming language into machine code, bytecode, or another programming language.\n\n## Key Stages of Compilation:\n\n1. **Lexical Analysis** - Breaking source code into tokens\n2. **Syntax Analysis** - Parsing tokens into an abstract syntax tree (AST)\n3. **Semantic Analysis** - Checking for semantic errors and type consistency\n4. **Intermediate Code Generation** - Creating an intermediate representation\n5. **Optimization** - Improving code efficiency\n6. **Code Generation** - Producing target machine code\n\n## Types of Compilers:\n\n- **Single-pass compilers** - Process source code in one pass\n- **Multi-pass compilers** - Process source code multiple times\n- **Just-in-Time (JIT) compilers** - Compile during program execution\n- **Cross-compilers** - Generate code for a different platform\n\n## Popular Co

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# Loading all pdf files from directory

dir_loader = DirectoryLoader(
    "../data/", # Path to directory
    glob="**/*.pdf", # Pattern matcher
    loader_cls=PyMuPDFLoader, # Loader Class
    show_progress=True,
    )

documents = dir_loader.load()
documents[:20]

100%|██████████| 2/2 [00:00<00:00,  3.15it/s]


[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': '../data/anthropic_genai.pdf', 'file_path': '../data/anthropic_genai.pdf', 'total_pages': 10, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-12T18:37:28+00:00', 'trapped': '', 'modDate': 'D:20250612183728Z', 'creationDate': '', 'page': 0}, page_content='Understanding \nGenerative AI\nWhat is Generative AI?\nGenerative AI refers to artificial intelligence systems \nthat can create new content rather than just analyzing \nexisting data.\nTraditional AI\nClassifies emails as spam or not spam\nGenerative AI\nCan write a completely new email for you\nThree pillars that made it possible\nAlgorithms\nTransformer architecture \n(2017) revolutionized \nprocessing of long text \npassages\nData explosion\nExplosion of digital data \n(websites, code repositories, \nand other text) provided raw \nmaterial for training these \nsystems\nComputing power\nMassive 

## Vector Embedding 

In [55]:
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] =  os.getenv("GOOGLE_API_KEY")

In [62]:
class EmbeddingManager:
    """Handles document embedding generation using Google Embedding"""

    def __init__(self, model_name: str = "models/gemini-embedding-001"):

        """
        Initialize the embedding Manager

        Args:
            model_name: Hugging face model name for Senctence Embedding

        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):

        """Load the Google Embedding Model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = GoogleGenerativeAIEmbeddings(model=self.model_name)
         #   print(f"Model loaded succefully embedding dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading embedding model: {self.model_name}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray: 
        """
        Generate embedding for a list of texts

        Args: 
            texts: List of text string to embed

        Returns: 
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generaing embedding for {len(texts)} texts...")
        embeddings = self.model.embed_documents(texts)
        np_embeddings = np.array(embeddings)
        print(f"Generated embedding with shape {np_embeddings.shape}")
        return embeddings
    
"""     def get_embedding_dimesion(self):
        #Getting model embedding dimesion
        if not self.model:
            raise ValueError("Model not loaded")
        
        return self.model.get_sentence_embedding_dimension() 
"""


# initiate the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loading embedding model: models/gemini-embedding-001


## Vector Store

In [82]:
class VectorStore:
    """Managing document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str="../vector_store"):
        """
        Initialize vector store

        Args: 
            collection_name:  Name of ChromaDB collection
            persist_directory: Path to the directory to persist vector store
            
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client= None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desctiption": "document for embedding rag"}
            )
            print(f"Vector store initialized: collection {self.collection_name}")
            print(f"Existing documents in collection {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddingss to vector store
            
            Args: 
               documents: List of Langchain documets
               embeddings: Corresponding vector embeddings for a specific document
        
        """

        if (len(documents) != len(embeddings)):
            raise ValueError("Number of embeddings should be equivalent to number of documents")
        
        print(f"Adding {len(documents)} to vectore store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generating ids
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content

            documents_text.append(doc.page_content)

            # Embedding

            embeddings_list.append(list(embedding))

           

        try: 
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

        except Exception as e:
            print(f"Failed to add documents and coresponding embedding to vectore store: {e}")


vector_store = VectorStore()
vector_store

Vector store initialized: collection pdf_documents
Existing documents in collection 0


In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(documents)[:50]

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts)


Generaing embedding for 50 texts...
Generated embedding with shape (50, 3072)


In [74]:
print(f"lenght of chunks is, {len(chunks)}")
print(f"lenght of chunks is, {len(embeddings)}")


lenght of chunks is, 50
lenght of chunks is, 50


In [83]:
vector_store.add_documents(chunks, embeddings)

Adding 50 to vectore store...


## RAG Retriever

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
            Intitalize RAG retriver with vectore store and embedding manager

            Args: 
                vector_store: Intstance of vector store containing the documents
                embedding_manager: Embedding manager for generating query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: Query to search for document
                top_k: Numbers of top results to return
                score_threshold: Minimun simularity score threshold

            Returns: 
                List of dictionaries containing documents and metadata
        """

        print(f"Retrieveing documents for query: {query}")
        print(f"Top k: {top_k}, Score Threshold: {score_threshold}")

        # Generate query embeddings
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # Seach in vector store

        try: 
            results = self.embedding_manager.collection.query(
                query_embeddings=[List(query_embedding)],
                n_results=top_k
            )

            retrieved_docs = []

            if (results["documents"] and results["documents"][0]):
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance
                    
                    if (similarity_score >= score_threshold):
                        retrieved_docs.append({
                            "id": doc_id,
                            "metadata": metadata,
                            "document": document,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents after simularity filtering") 

            else: 
                print("No document found")  

            return  retrieved_docs 

        except Exception as e:
            print(f"Error query vector store {e}") 




